# H2 · Per-building table and SQL counting agent (O2 / O3)

Task **H2** from `Essential_task_delegation/Next_Steps_Roadmap.md`. A small LLM answers building-counting questions by **writing SQL** over a per-building table, instead of reading pixels. Exact queries ("how many destroyed buildings within 100 px of a damaged one") become possible, which is the base for DisasTeller's offline back end (T2) and the demo (T3).

**What it does**
1. **Segments** the benchmark after-images with R9 (standard prediction, no TTA: this is the mask H1 V1's tuned counter used) and builds **`buildings.sqlite`**: one row per building blob (image, ID, class, area, centroid, bounding box, distance to the nearest destroyed building).
   - A blob is an 8-connected component of one class, the same definition DisasterM3 used for its counting answers.
   - Blobs smaller than H1 V1's tuned minimum area are not stored (intact < 24 px; damaged and destroyed keep every blob), so `COUNT(*)` reproduces V1's counts.
2. **Checks the table against H1 V1** on every counting question: the table's count must equal V1's `count`.
3. **Samples 100 BDC questions** (stratified by class, fixed seed) and for each LLM in `LLM_IDS`:
   - gives the model the schema and the question, and lets it write one SQL query;
   - runs the query on a **read-only** connection (SELECT only, with a time limit and up to 2 retries on errors);
   - picks the option nearest to the result, as V1 does, and has the model phrase a one-sentence answer.
4. **Compares** the agent with H1 V1 and with an "oracle" that runs the correct SQL by hand, on the same questions.

**What to expect.** The table holds the same R9 blobs V1 counted, so even a perfect agent scores about V1's level (about 22% overall, about 27% on optical images). H2 does not make counting more accurate; it makes the counts **queryable** and shows whether a small local LLM writes reliable SQL.

**How to run on Kaggle**
1. Settings: **GPU T4 ×2**, internet ON (T4 ×1 is enough for the 3B model only; the 7B needs both GPUs).
2. Inputs:
   - **`redwannewaz/disasterm3-bench-mirror-v2`** (`benchmark_release.json` and `test_images/`);
   - **`h1-v1-facts`** (the H1 V1 output; this notebook reads its `H1_v1_per_question.csv`).
3. Secret `HF_TOKEN` (R9's repo `AbrarAlam/disasterm3-d4-R9`), as in H1 V1. Or attach a dataset with `R9/final_model.pth`.
4. Run with `SMOKE_TEST = True` first (6 questions, a few images, one LLM; about 10 minutes with the model download). Then Save & Run All.
5. Outputs in `/kaggle/working/H2/`: `buildings.sqlite`, `agent_eval.csv`, `agent_summary.csv`, `table_check.csv`, `config.json`, `README.md`.

**Done when:** `buildings.sqlite` exists and matches V1's counts, `agent_eval.csv` covers the 100 questions for each LLM, and the H2-against-V1 comparison is printed.

In [ ]:
# ── Cell 1: Environment Setup (Run-All safe, identical to the D4 notebooks and H1 V1) ──
# Kaggle's image moved to Python 3.13, which has no torch 2.4.1 wheel. Use Kaggle's own
# torch/torchvision/opencv and pin only the libraries whose API this notebook depends on.
import importlib.metadata as _md
import os, sys, subprocess

PINS = {
    "segmentation-models-pytorch": "0.3.4",   # baseline version (checkpoint key layout)
    "albumentations": "1.4.21",               # 1.x API (PadIfNeeded value=/mask_value=)
}

def _mismatched():
    out = []
    for pkg, want in PINS.items():
        try:
            have = _md.version(pkg)
        except _md.PackageNotFoundError:
            have = "not installed"
        if have != want:
            out.append(f"{pkg}: {have} → {want}")
    return out

if _mismatched():
    print("⏳ Installing pinned versions:")
    for _m in _mismatched():
        print(f"   {_m}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *(f"{k}=={v}" for k, v in PINS.items()), "huggingface_hub[hf_xet]"],
                   check=True)                # pip failure stops the run here
    if _mismatched():
        raise RuntimeError(f"Pins still wrong after install: {_mismatched()}")
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
        # background "Save & Run All": nothing imported yet, so new versions load on first import
        print("✓ Dependencies installed (background run) — continuing without restart.")
    else:
        raise SystemExit("✓ Dependencies installed. RESTART THE KERNEL NOW "
                         "(Run → Restart & clear cell outputs), then Run All again.")
else:
    print("✓ All pinned versions already installed — proceeding.")

import torch
print(f"  torch {torch.__version__}, python {sys.version.split()[0]}")

In [ ]:
# ── Cell 2: Configuration ──────────────────────────────────────────────────
import gc
import glob
import json
import os
import re
import sqlite3
import time
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

OUT_DIR = Path("/kaggle/working/H2")
SMOKE_TEST = False                   # True = 6 questions, a few images, the first LLM only
N_QUESTIONS = 100                    # roadmap: test on 100 BDC questions
SAMPLE_SENSOR = "Optical"            # "Optical", "SAR" or None. SAR truth can differ from the optical twin's counts
SEED = 42
FULL_TABLE = True                    # True = every benchmark after-image (about 2,650, +10 min); False = only the sampled questions' images
USE_CATEGORY_HINT = True             # also give the agent the question's category ("Intact building counting.") next to its text
GSD_M_PER_PX = None                  # metres per pixel if known (DisasterM3 gives none in our files). None = distances in pixels only
RADIUS_PX = 100                      # radius of the demo question below, when GSD_M_PER_PX is None
RADIUS_M = 100                       # ... and in metres when it is set
LLM_IDS = ["Qwen/Qwen2.5-Coder-3B-Instruct",                # small local text LLM (one T4)
           "AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP"]     # our fine-tuned 7B, text only (needs both T4s)
MAX_SQL_RETRIES = 2                  # extra attempts after a failed query
MODEL_NAME = "R9"                    # the D4 champion recipe
HF_USER = "AbrarAlam"                # checkpoint: {HF_USER}/disasterm3-d4-R9/final_model.pth
CKPT_PATH = None                     # set a path to skip the search, e.g. "/kaggle/input/my-ckpts/R9/final_model.pth"
NUM_CLASSES = 4                      # 0=Background, 1=Intact, 2=Damaged, 3=Destroyed
# H1 V1's tuned rule (its config.json: standard mask, no closing): blobs smaller than this are not counted
MIN_AREA = {1: 24, 2: 0, 3: 0}
TASK = "Building Damage Counting"
CLASS_OF = {"Intact building counting.": "intact", "Damaged building counting.": "damaged",
            "Totally-destroyed building counting.": "destroyed"}
CLASS_NAME = {1: "intact", 2: "damaged", 3: "destroyed"}
CLASS_ID = {v: k for k, v in CLASS_NAME.items()}
SIX_TASKS = {"Disaster Bearing Bodies Recognition", "Building Damage Counting", "Disaster Type Recognition",
             "Road Damage Counting", "Disaster Scene Recognition", "Relational Reasoning"}

if SMOKE_TEST:
    N_QUESTIONS, FULL_TABLE, LLM_IDS = 6, False, LLM_IDS[:1]

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def find_one(pattern, why):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"No {pattern} under /kaggle/input: {why}")
    return Path(hits[0])

def find_checkpoint(name):
    # Order: CKPT_PATH, then a /kaggle/input folder named R9 / d4_R9 / R9-results, then Hugging Face.
    if CKPT_PATH:
        return CKPT_PATH
    wanted = {name.lower(), f"d4_{name}".lower(), f"{name}-results".lower()}
    hits = [p for p in glob.glob("/kaggle/input/**/*.pth", recursive=True)
            if any(part.lower() in wanted for part in Path(p).parts)]
    hits.sort(key=lambda p: Path(p).name != "final_model.pth")   # prefer final_model.pth
    if hits:
        return hits[0]
    repo, errors = f"{HF_USER}/disasterm3-d4-{name}", []
    for filename in ("final_model.pth", "last.pth"):   # last.pth holds the same epoch-40 weights
        try:
            return hf_hub_download(repo, filename, token=hf_token())
        except Exception as e:
            errors.append(f"{filename}: {type(e).__name__}")
    raise FileNotFoundError(f"No checkpoint for {name}: nothing under /kaggle/input and {repo} gave {errors}.")

BENCH_JSON = find_one("benchmark_release.json", "attach redwannewaz/disasterm3-bench-mirror-v2")
v1_hits = sorted(glob.glob("/kaggle/input/**/H1_v1_per_question.csv", recursive=True))
if not v1_hits and not SMOKE_TEST:
    raise FileNotFoundError("Attach h1-v1-facts (the H1 V1 output): its H1_v1_per_question.csv is the comparison.")
V1 = pd.read_csv(v1_hits[0]) if v1_hits else None
CKPT = find_checkpoint(MODEL_NAME)
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ H2{'  [SMOKE TEST]' if SMOKE_TEST else ''}: {MODEL_NAME} from {CKPT}")
print(f"  Benchmark: {BENCH_JSON.parent}")
print(f"  H1 V1 per-question results: {'found, ' + str(len(V1)) + ' rows' if V1 is not None else 'NOT attached (smoke test only)'}")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'} x{torch.cuda.device_count()}")
print(f"  LLMs: {LLM_IDS}")

In [ ]:
# ── Cell 3: R9, standard prediction (as in H1 V1: pad to a multiple of 32, argmax) ─────────
import segmentation_models_pytorch as smp
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2

NORMALIZE = A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
val_transform = A.Compose([NORMALIZE, ToTensorV2()])   # E-protocol: full native image, no resize

def load_model(path):
    # D4 checkpoints: U-Net + ResNet-50, 4 classes. Accepts a state dict or a resume checkpoint.
    state = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=3, classes=NUM_CLASSES)
    model.load_state_dict(state)
    return model.to(DEVICE).eval()

@torch.no_grad()
def segment_std(model, path):
    # 0-3 mask (uint8) at the image's own size, or None if the file cannot be read
    image = cv2.imread(str(path))
    if image is None:
        return None
    x = val_transform(image=cv2.cvtColor(image, cv2.COLOR_BGR2RGB))["image"].unsqueeze(0).to(DEVICE)
    h, w = x.shape[-2:]
    ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
    if ph or pw:
        x = F.pad(x, (0, pw, 0, ph), mode="reflect")
    with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
        logits = model(x)[..., :h, :w].float()
    return logits.argmax(1)[0].cpu().numpy().astype(np.uint8)

print("✓ Segmentation helpers ready")

In [ ]:
# ── Cell 4: The benchmark BDC questions and the 100-question sample ─────────────────────────
def name_of(path):
    return Path(str(path).replace("\\", "/")).name

with open(BENCH_JSON, encoding="utf-8") as f:
    bench = json.load(f)
bdc = [e for e in bench if e.get("task") == TASK]
REQUIRED = ("prompts", "options_list", "ground_truth", "ground_truth_option", "cls_description",
            "post_image_type", "post_image_path")
lacking = [k for k in REQUIRED if k not in bdc[0]]
if lacking:
    raise KeyError(f"BDC entries lack {lacking}; their fields are {sorted(bdc[0])}")
for i, e in enumerate(bdc):                      # ids as in the 499A scorer and H1 V1: subset name + index in benchmark order
    e["id"] = f"building_damage_counting_{i}"
    e["image"] = name_of(e["post_image_path"]).replace("_sar", "")   # SAR questions use the optical twin
unknown = Counter(e["cls_description"] for e in bdc if e["cls_description"] not in CLASS_OF)
assert not unknown, f"Unexpected BDC classes: {dict(unknown)}"

QDF = pd.DataFrame([{"id": e["id"], "cls": CLASS_OF[e["cls_description"]], "category": e["cls_description"],
                     "sensor": e["post_image_type"], "image": e["image"], "truth": e["ground_truth"],
                     "truth_option": e["ground_truth_option"], "question": e["prompts"],
                     "options": list(e["options_list"])} for e in bdc])
print(f"BDC questions: {len(QDF):,}  {dict(QDF['cls'].value_counts())}  {dict(QDF['sensor'].value_counts())}")

def sample_questions(df, n, sensor, seed):
    # n questions, split evenly over the three classes, fixed seed. Remainder goes to intact, then damaged.
    pool = df if sensor is None else df[df["sensor"] == sensor]
    base, extra = divmod(n, 3)
    parts = []
    for k, cls in enumerate(("intact", "damaged", "destroyed")):
        want = base + (1 if k < extra else 0)
        g = pool[pool["cls"] == cls]
        parts.append(g.sample(min(want, len(g)), random_state=seed))
    return pd.concat(parts).sort_values("id", key=lambda s: s.str.rsplit("_", n=1).str[1].astype(int)).reset_index(drop=True)

SAMPLE = sample_questions(QDF, N_QUESTIONS, SAMPLE_SENSOR, SEED)
print(f"Sample: {len(SAMPLE)} questions ({SAMPLE_SENSOR or 'all sensors'}), {SAMPLE['image'].nunique()} images, "
      f"{dict(SAMPLE['cls'].value_counts())}")
print("\nThree question texts, as the agent will see them (check that they name the class; if so, set USE_CATEGORY_HINT = False):")
for _, r in SAMPLE.head(3).iterrows():
    print(f"  [{r['category']}] {r['question']!r}")

# Benchmark images, indexed by file name wherever the attached datasets keep them (as in H1 V1)
bench_index = {}
for p in glob.glob("/kaggle/input/**/test_images/**/*.png", recursive=True):
    bench_index.setdefault(Path(p).name, p)
all_after = sorted({name_of(e["post_image_path"]).replace("_sar", "")
                    for e in bench if e.get("task") in SIX_TASKS and e.get("post_image_path")})
IMAGES = all_after if FULL_TABLE else sorted(SAMPLE["image"].unique())
found = sum(n in bench_index for n in IMAGES)
if found < 0.99 * len(IMAGES):
    raise FileNotFoundError(f"Only {found:,} of {len(IMAGES):,} benchmark after-images found under /kaggle/input. "
                            "Attach redwannewaz/disasterm3-bench-mirror-v2 (its test_images/ folder) and run again.")
print(f"\n✓ Images to segment: {len(IMAGES):,} ({found:,} found)")

In [ ]:
# ── Cell 5: The per-building table (pure functions) ─────────────────────────────────────────
from scipy.spatial import cKDTree

# (column, SQL type, comment). The comments are shown to the LLM, so they are written for it.
COLUMNS = [
    ("image", "TEXT", "after-image file name"),
    ("building_id", "INTEGER", "building number within that image"),
    ("class_id", "INTEGER", "1 = intact, 2 = damaged, 3 = destroyed"),
    ("class_name", "TEXT", "'intact', 'damaged' or 'destroyed'"),
    ("area_px", "INTEGER", "building area in pixels"),
    ("centroid_x", "REAL", "building centre, pixels from the left edge"),
    ("centroid_y", "REAL", "building centre, pixels from the top edge (y grows downwards)"),
    ("bbox_x", "INTEGER", "bounding box left"),
    ("bbox_y", "INTEGER", "bounding box top"),
    ("bbox_w", "INTEGER", "bounding box width"),
    ("bbox_h", "INTEGER", "bounding box height"),
    ("dist_to_nearest_destroyed_px", "REAL",
     "centre-to-centre distance in pixels to the nearest OTHER destroyed building in the same image; NULL if there is none"),
]
if GSD_M_PER_PX:
    COLUMNS.append(("dist_to_nearest_destroyed_m", "REAL", "the same distance in metres"))

def create_schema(conn):
    cols = ",\n  ".join(f"{n} {t}" for n, t, _ in COLUMNS)
    conn.execute(f"CREATE TABLE buildings (\n  {cols},\n  PRIMARY KEY (image, building_id))")
    conn.execute("CREATE TABLE images (image TEXT PRIMARY KEY, width INTEGER, height INTEGER)")
    conn.execute("CREATE INDEX idx_buildings_class ON buildings (image, class_name)")

def schema_prompt_text():
    lines = ["TABLE buildings  -- one row per building (an 8-connected blob of one class in the U-Net mask)"]
    lines += [f"  {n} {t},  -- {c}" for n, t, c in COLUMNS]
    lines += ["TABLE images  -- one row per image", "  image TEXT, width INTEGER, height INTEGER"]
    return "\n".join(lines)

def add_nearest_destroyed(blobs):
    # Centre-to-centre distance to the nearest OTHER destroyed blob, in pixels (None if there is none)
    for b in blobs:
        b["dist_to_nearest_destroyed_px"] = None
        if GSD_M_PER_PX:
            b["dist_to_nearest_destroyed_m"] = None
    dest = [b for b in blobs if b["class_id"] == 3]
    if not dest:
        return
    tree = cKDTree(np.array([[b["centroid_x"], b["centroid_y"]] for b in dest]))
    for b in blobs:
        p = (b["centroid_x"], b["centroid_y"])
        if b["class_id"] == 3:
            if len(dest) < 2:
                continue
            d = float(tree.query(p, k=2)[0][1])      # the nearest hit is the blob itself, at distance 0
        else:
            d = float(tree.query(p, k=1)[0])
        b["dist_to_nearest_destroyed_px"] = round(d, 1)
        if GSD_M_PER_PX:
            b["dist_to_nearest_destroyed_m"] = round(d * GSD_M_PER_PX, 1)

def blobs_of(pred):
    # Building rows for one 0-3 mask: 8-connected blobs per class, minus blobs under MIN_AREA (H1 V1's counting rule)
    out = []
    for cls, cname in CLASS_NAME.items():
        n, _, stats, cents = cv2.connectedComponentsWithStats((pred == cls).astype(np.uint8), connectivity=8)
        for k in range(1, n):
            area = int(stats[k, cv2.CC_STAT_AREA])
            if area < MIN_AREA[cls]:
                continue
            out.append({"class_id": cls, "class_name": cname, "area_px": area,
                        "centroid_x": round(float(cents[k][0]), 1), "centroid_y": round(float(cents[k][1]), 1),
                        "bbox_x": int(stats[k, cv2.CC_STAT_LEFT]), "bbox_y": int(stats[k, cv2.CC_STAT_TOP]),
                        "bbox_w": int(stats[k, cv2.CC_STAT_WIDTH]), "bbox_h": int(stats[k, cv2.CC_STAT_HEIGHT])})
    for i, b in enumerate(out, 1):
        b["building_id"] = i
    add_nearest_destroyed(out)
    return out

def insert_image(conn, name, height, width, blobs):
    conn.execute("INSERT INTO images VALUES (?, ?, ?)", (name, width, height))
    names = [n for n, _, _ in COLUMNS]
    conn.executemany(f"INSERT INTO buildings ({', '.join(names)}) VALUES ({', '.join('?' * len(names))})",
                     [tuple(name if n == "image" else b[n] for n in names) for b in blobs])

def nearest_option(n, options):
    return "ABCDE"[min(range(len(options)), key=lambda i: abs(options[i] - n))]

# Self-checks: a diagonal touch is one blob (8-connected), blobs under the minimum are dropped,
# and the distance to the nearest destroyed building leaves out the building itself.
_m = np.zeros((200, 200), np.uint8)
_m[10:20, 10:20] = 1                    # intact, 100 px
_m[20, 20] = 1                          # touches that blob on a diagonal: still one blob
_m[100:102, 100:102] = 1                # intact, 4 px: under MIN_AREA[1]
_m[50:60, 50:60] = 3                    # destroyed
_m[50:60, 150:160] = 3                  # destroyed
_b = blobs_of(_m)
assert [b["class_name"] for b in _b].count("intact") == 1 and sum(b["class_id"] == 3 for b in _b) == 2
_dd = [b["dist_to_nearest_destroyed_px"] for b in _b if b["class_id"] == 3]
assert _dd == [100.0, 100.0], _dd
assert nearest_option(52, [10, 40, 60, 80, 100]) == "C"
print("✓ Table helpers ready; self-checks passed")

## 1. Segment the images and build `buildings.sqlite`
R9's standard (non-TTA) prediction, one pass per image. H1 V1 tuned its counter on the standard mask for all three classes, so the table holds the same blobs.

In [ ]:
# ── Cell 6: Segment with R9, write buildings.sqlite ─────────────────────────────────────────
t_start = time.time()
DB_PATH = OUT_DIR / "buildings.sqlite"
if DB_PATH.exists():
    DB_PATH.unlink()

model = load_model(CKPT)
conn = sqlite3.connect(DB_PATH)
create_schema(conn)
missing = []
for name in tqdm(IMAGES, desc="Images"):
    path = bench_index.get(name)
    pred = segment_std(model, path) if path else None
    if pred is None:
        missing.append(name)
        continue
    insert_image(conn, name, pred.shape[0], pred.shape[1], blobs_of(pred))
conn.commit()
n_rows = conn.execute("SELECT COUNT(*) FROM buildings").fetchone()[0]
n_imgs = conn.execute("SELECT COUNT(*) FROM images").fetchone()[0]
by_class = dict(conn.execute("SELECT class_name, COUNT(*) FROM buildings GROUP BY class_name").fetchall())
conn.close()
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"✓ buildings.sqlite: {n_rows:,} buildings in {n_imgs:,} images {by_class} "
      f"({DB_PATH.stat().st_size / 1e6:.1f} MB, {(time.time() - t_start) / 60:.1f} min)")
if missing:
    print(f"⚠ Not read ({len(missing)}; their questions count as wrong): {missing[:5]}")

In [ ]:
# ── Cell 7: Check the table against H1 V1 (its counts are the answer key's blob counts of R9's mask) ──
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
table_counts = {(im, cn): n for im, cn, n in
                conn.execute("SELECT image, class_name, COUNT(*) FROM buildings GROUP BY image, class_name")}
in_table = {r[0] for r in conn.execute("SELECT image FROM images")}
conn.close()

def table_count(image, cls):
    return table_counts.get((image, cls), 0) if image in in_table else np.nan

QDF["table_count"] = [table_count(im, c) for im, c in zip(QDF["image"], QDF["cls"])]
QDF["oracle_answer"] = [nearest_option(n, o) if n == n else "" for n, o in zip(QDF["table_count"], QDF["options"])]
QDF["oracle_correct"] = QDF["oracle_answer"] == QDF["truth_option"]
covered = QDF["table_count"].notna()
check = pd.DataFrame([{"questions_in_table": int(covered.sum()), "oracle_accuracy_%": round(100 * QDF.loc[covered, "oracle_correct"].mean(), 2)}])
if V1 is not None:
    v = QDF.merge(V1[["id", "count", "correct"]].rename(columns={"count": "v1_count", "correct": "v1_correct"}), on="id", how="left")
    both = v["table_count"].notna() & v["v1_count"].notna()
    check["same_count_as_V1_%"] = round(100 * (v.loc[both, "table_count"] == v.loc[both, "v1_count"]).mean(), 2)
    check["V1_accuracy_on_same_%"] = round(100 * v.loc[covered, "v1_correct"].astype(float).mean(), 2)
    print("Table against H1 V1 on every BDC question in the table:")
else:
    print("H1 V1 results not attached: only the oracle accuracy can be shown.")
print(check.to_string(index=False))
check.to_csv(OUT_DIR / "table_check.csv", index=False)
if V1 is not None and check["same_count_as_V1_%"].iloc[0] < 99:
    print("⚠ The table should reproduce V1's counts (same checkpoint, standard mask, same minimum areas). "
          "A gap above 1% means a different checkpoint or mask: look before trusting the agent comparison.")

## 2. The SQL agent
The model sees the schema, three worked examples and one question. It must answer with one SELECT query, which runs on a read-only connection. A failed query is sent back with its error, up to `MAX_SQL_RETRIES` times.

In [ ]:
# ── Cell 8: Agent prompt, SQL guard, read-only execution, retry loop (pure functions) ─────────
FORBIDDEN = re.compile(r"\b(insert|update|delete|drop|alter|create|attach|detach|pragma|vacuum|reindex)\b", re.I)
M = "metres" if GSD_M_PER_PX else "pixels"
DIST_COL = "dist_to_nearest_destroyed_m" if GSD_M_PER_PX else "dist_to_nearest_destroyed_px"

SYSTEM_PROMPT = (
    "You answer questions about ONE satellite image of a disaster area by writing a single SQLite SELECT query "
    "over this schema.\n\n" + schema_prompt_text() + "\n\n"
    "Rules:\n"
    "- Reply with the SQL only: one SELECT statement, no explanation, no markdown.\n"
    "- Always filter on the image named in the message: WHERE image = '<that file name>'.\n"
    "- Counting questions: use COUNT(*) on the matching class_name.\n"
    f"- Distances and areas are in {'metres (areas in pixels)' if GSD_M_PER_PX else 'pixels'}.\n\n"
    "Examples (the file names are placeholders):\n"
    "Image: example_a.png\nQuestion: How many destroyed buildings are there?\n"
    "SQL: SELECT COUNT(*) FROM buildings WHERE image = 'example_a.png' AND class_name = 'destroyed'\n\n"
    "Image: example_b.png\nQuestion: How many intact buildings are within 150 of a destroyed building?\n"
    f"SQL: SELECT COUNT(*) FROM buildings WHERE image = 'example_b.png' AND class_name = 'intact' AND {DIST_COL} <= 150\n\n"
    "Image: example_c.png\nQuestion: What is the total area of the damaged buildings?\n"
    "SQL: SELECT COALESCE(SUM(area_px), 0) FROM buildings WHERE image = 'example_c.png' AND class_name = 'damaged'"
)

def user_prompt(question, image, category=None):
    hint = f"\nCategory: {category}" if USE_CATEGORY_HINT and category else ""
    return f"Image: {image}\nQuestion: {question}{hint}\nSQL:"

def clean_sql(text):
    t = text.strip()
    m = re.search(r"```(?:sql)?\s*(.*?)```", t, re.S | re.I)
    if m:
        t = m.group(1).strip()
    t = re.sub(r"^\s*SQL:\s*", "", t, flags=re.I)
    return t.strip().rstrip(";").strip()

def check_sql(sql):
    if not re.match(r"^(select|with)\b", sql, re.I):
        return "the reply is not a SELECT statement"
    if ";" in sql:
        return "more than one statement"
    if FORBIDDEN.search(sql):
        return "a forbidden keyword"
    return None

def open_readonly(path):
    c = sqlite3.connect(f"file:{path}?mode=ro", uri=True)
    c.execute("PRAGMA query_only = ON")
    return c

def run_sql(conn, sql, timeout_s=5):
    t0 = time.time()
    conn.set_progress_handler(lambda: 1 if time.time() - t0 > timeout_s else 0, 100000)
    try:
        return conn.execute(sql).fetchmany(50), None
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"
    finally:
        conn.set_progress_handler(None, 0)

def as_number(rows):
    if rows is not None and len(rows) == 1 and len(rows[0]) == 1 and isinstance(rows[0][0], (int, float)):
        return float(rows[0][0]), None
    return None, "the query must return exactly one number"

def agent_answer(llm, conn, question, image, category=None):
    # One question -> {sql, n, ok, attempts, error}. A failed query goes back to the model with its error.
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt(question, image, category)}]
    err, sql = "no attempt", ""
    for attempt in range(MAX_SQL_RETRIES + 1):
        raw = llm.chat(messages, max_new_tokens=200)
        sql = clean_sql(raw)
        err, n = check_sql(sql), None
        if err is None:
            rows, err = run_sql(conn, sql)
            if err is None:
                n, err = as_number(rows)
        if err is None:
            return {"sql": sql, "n": n, "ok": True, "attempts": attempt + 1, "error": ""}
        messages += [{"role": "assistant", "content": raw},
                     {"role": "user", "content": f"That query failed: {err}. Write one corrected SELECT query. SQL:"}]
    return {"sql": sql, "n": None, "ok": False, "attempts": MAX_SQL_RETRIES + 1, "error": err}

def phrase_answer(llm, question, image, n):
    messages = [{"role": "system", "content": "You report the result of a database query. Answer in one short sentence, "
                                              "using only the number you are given."},
                {"role": "user", "content": f"Image: {image}\nQuestion: {question}\nQuery result: {n:g}"}]
    return llm.chat(messages, max_new_tokens=60).strip().replace("\n", " ")

print("✓ Agent helpers ready")

In [ ]:
# ── Cell 9: A local LLM behind chat(messages) -> text (greedy; text only, no images) ─────────
class LocalLLM:
    def __init__(self, model_id):
        import transformers
        self.id = model_id
        is_vl = "vl" in model_id.lower()
        if is_vl:   # our fine-tuned Qwen2.5-VL: only its language side is used, so no images are passed
            self.tmpl = transformers.AutoProcessor.from_pretrained(model_id, token=hf_token())
            self.tok = self.tmpl.tokenizer
            cls = transformers.Qwen2_5_VLForConditionalGeneration
        else:
            self.tmpl = self.tok = transformers.AutoTokenizer.from_pretrained(model_id, token=hf_token())
            cls = transformers.AutoModelForCausalLM
        self.model = cls.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto", token=hf_token()).eval()

    @torch.no_grad()
    def chat(self, messages, max_new_tokens=200):
        text = self.tmpl.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tok(text, return_tensors="pt").to(self.model.device)
        out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        return self.tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    def free(self):
        del self.model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("✓ LLM wrapper ready")

## 3. Answer the sampled BDC questions
Per LLM: write SQL, run it, take the option nearest to the result, phrase a sentence. Each answer is appended to `agent_eval_partial.jsonl` as it comes, so a re-run in the same session continues where it stopped.

In [ ]:
# ── Cell 10: Run the agent on the sample, once per LLM ──────────────────────────────────────
PARTIAL = OUT_DIR / "agent_eval_partial.jsonl"
done = {}
if PARTIAL.exists():
    for line in PARTIAL.read_text().splitlines():
        r = json.loads(line)
        done[(r["llm"], r["id"])] = r

conn = open_readonly(DB_PATH)
t_start = time.time()
for model_id in LLM_IDS:
    todo = [r for _, r in SAMPLE.iterrows() if (model_id, r["id"]) not in done]
    if not todo:
        continue
    if "vl" in model_id.lower() and torch.cuda.device_count() < 2:
        print(f"⚠ Skipping {model_id}: the 7B needs two T4s, this session has {torch.cuda.device_count()}.")
        continue
    print(f"\nLoading {model_id} ...")
    llm = LocalLLM(model_id)
    for r in tqdm(todo, desc=model_id.split("/")[-1]):
        res = agent_answer(llm, conn, r["question"], r["image"], r["category"])
        answer = nearest_option(res["n"], r["options"]) if res["ok"] else ""
        row = {"id": r["id"], "llm": model_id, "class": r["cls"], "sensor": r["sensor"], "image": r["image"],
               "truth": r["truth"], "truth_option": r["truth_option"], "question": r["question"],
               "sql": res["sql"], "sql_ok": res["ok"], "attempts": res["attempts"], "error": res["error"],
               "n": res["n"], "h2_answer": answer, "h2_correct": answer == r["truth_option"],
               "phrased": phrase_answer(llm, r["question"], r["image"], res["n"]) if res["ok"] else ""}
        done[(model_id, r["id"])] = row
        with open(PARTIAL, "a") as f:
            f.write(json.dumps(row) + "\n")
    llm.free()
    del llm
conn.close()
print(f"\n✓ {len(done)} answers ({(time.time() - t_start) / 60:.1f} min)")

In [ ]:
# ── Cell 11: agent_eval.csv, the comparison with H1 V1 and with the oracle ───────────────────
rows = [r for (llm, qid), r in done.items() if llm in LLM_IDS and qid in set(SAMPLE["id"])]
EVAL = pd.DataFrame(rows)
if EVAL.empty:
    raise RuntimeError("No answers were produced: see the messages above.")
ref = QDF.set_index("id")
EVAL["ref_count"] = EVAL["id"].map(ref["table_count"])
EVAL["oracle_answer"] = EVAL["id"].map(ref["oracle_answer"])
EVAL["oracle_correct"] = EVAL["id"].map(ref["oracle_correct"])
if V1 is not None:
    v1 = V1.set_index("id")
    EVAL["v1_count"] = EVAL["id"].map(v1["count"])
    EVAL["v1_answer"] = EVAL["id"].map(v1["answer"])
    EVAL["v1_correct"] = EVAL["id"].map(v1["correct"])
EVAL["same_count_as_table"] = EVAL["n"] == EVAL["ref_count"]
EVAL.to_csv(OUT_DIR / "agent_eval.csv", index=False)

def pct(s):
    return round(100 * s.astype(float).mean(), 1)

def summarize(g):
    out = {"questions": len(g), "sql_valid_%": pct(g["sql_ok"]), "retried_%": pct(g["attempts"] > 1),
           "count_equals_table_%": pct(g["same_count_as_table"]), "H2_accuracy_%": pct(g["h2_correct"]),
           "oracle_accuracy_%": pct(g["oracle_correct"])}
    if "v1_correct" in g:
        out["V1_accuracy_%"] = pct(g["v1_correct"])
    return out

SUMMARY = pd.DataFrame([{"llm": llm, **summarize(g)} for llm, g in EVAL.groupby("llm", sort=False)])
BY_CLASS = pd.DataFrame([{"llm": llm, "class": c, **summarize(g)} for (llm, c), g in EVAL.groupby(["llm", "class"], sort=False)])
SUMMARY.to_csv(OUT_DIR / "agent_summary.csv", index=False)
print("Per LLM (one row per model, the same sampled questions):")
print(SUMMARY.to_string(index=False))
print("\nPer class:")
print(BY_CLASS.to_string(index=False))
if V1 is not None:
    full = V1.merge(QDF[["id", "sensor"]].rename(columns={"sensor": "s"}), on="id")
    print(f"\nFor context, H1 V1 on all 4,982 questions: {pct(V1['correct'])}% "
          f"(optical {pct(V1.loc[V1['sensor'] == 'Optical', 'correct'])}%); base Qwen2.5-VL-7B 34.2%. "
          f"{len(SAMPLE)} questions give about ±{round(100 * 1.96 * 0.5 / len(SAMPLE) ** 0.5, 1)} points of noise at 95% (worst case).")
bad = EVAL[~EVAL["sql_ok"]]
if len(bad):
    print(f"\nFailed queries ({len(bad)}), first 3:")
    for _, r in bad.head(3).iterrows():
        print(f"  {r['llm'].split('/')[-1]} · {r['id']}: {r['error']} · {r['sql'][:120]!r}")

In [ ]:
# ── Cell 12: Three free-form questions, to show what the table allows beyond counting ────────
if not EVAL.empty:
    demo_llm = LLM_IDS[0]
    demo_image = SAMPLE["image"].iloc[0]
    radius = f"{RADIUS_M} metres" if GSD_M_PER_PX else f"{RADIUS_PX} pixels"
    demo_qs = [f"How many intact buildings are within {radius} of a destroyed building?",
               "What is the area of the largest damaged building?",
               "How many destroyed buildings are in the left half of the image (centroid_x below half the image width)?"]
    conn = open_readonly(DB_PATH)
    llm = LocalLLM(demo_llm)
    DEMO = []
    for q in demo_qs:
        res = agent_answer(llm, conn, q, demo_image)
        DEMO.append({"image": demo_image, "question": q, **res})
        print(f"Q: {q}\n   SQL: {res['sql']}\n   result: {res['n']} (ok={res['ok']}, attempts={res['attempts']})\n")
    llm.free()
    conn.close()
    pd.DataFrame(DEMO).to_csv(OUT_DIR / "demo_queries.csv", index=False)

In [ ]:
# ── Cell 13: README.md and config.json ──────────────────────────────────────────────────────
def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

tbl_match = check["same_count_as_V1_%"].iloc[0] if "same_count_as_V1_%" in check else None
config = {"task": "H2", "smoke_test": SMOKE_TEST, "model": MODEL_NAME, "checkpoint": str(CKPT),
          "mask": "R9 standard prediction (no TTA)", "min_area": {CLASS_NAME[k]: v for k, v in MIN_AREA.items()},
          "images_in_table": n_imgs, "buildings_in_table": n_rows, "full_table": FULL_TABLE,
          "gsd_m_per_px": GSD_M_PER_PX, "n_questions": len(SAMPLE), "sample_sensor": SAMPLE_SENSOR, "seed": SEED,
          "use_category_hint": USE_CATEGORY_HINT, "llms": LLM_IDS, "max_sql_retries": MAX_SQL_RETRIES,
          "same_count_as_V1_pct": None if tbl_match is None else float(tbl_match),
          "run_time_min": round((time.time() - t_start) / 60, 1)}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1, default=float), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    "# H2: per-building table and SQL counting agent", "",
    f"R9 standard mask on {n_imgs:,} benchmark after-images, {n_rows:,} buildings in `buildings.sqlite`"
    f"{' (SMOKE TEST)' if SMOKE_TEST else ''}. {len(SAMPLE)} BDC questions "
    f"({SAMPLE_SENSOR or 'all sensors'}, seed {SEED}, split evenly over the classes).", "",
    "## Result", "", as_markdown(SUMMARY), "",
    "- `sql_valid_%`: the query ran and returned one number (after up to "
    f"{MAX_SQL_RETRIES} retries).",
    "- `count_equals_table_%`: the agent's number equals a hand-written `COUNT(*)` for that image and class.",
    "- `oracle_accuracy_%`: the hand-written query, i.e. the best any agent can do on this table.",
    "- `V1_accuracy_%`: H1 V1 on the same questions.", "",
    "## Table against H1 V1", "", as_markdown(check), "",
    "## Per class", "", as_markdown(BY_CLASS), "",
    "## Notes", "",
    "- The table holds the same R9 blobs H1 V1 counted, so H2 cannot beat the oracle: it makes counts queryable, "
    "not more accurate.",
    "- Distances are centre-to-centre, in pixels" + (f" and metres (GSD {GSD_M_PER_PX} m/px, an assumption)" if GSD_M_PER_PX else
                                                  " (DisasterM3 gives no ground resolution in our files)") + ".",
    f"- The agent saw the question text{' and its category line' if USE_CATEGORY_HINT else ''}, the image name, "
    "and no options.", "",
    "## Files", "",
    "- `buildings.sqlite` (tables `buildings`, `images`), `agent_eval.csv` (one row per LLM and question), "
    "`agent_summary.csv`, `table_check.csv`, `demo_queries.csv`, `config.json`, `agent_eval_partial.jsonl`",
]), encoding="utf-8")
print(f"✓ Wrote the outputs to {OUT_DIR}")
print("Next: publish this output as a Kaggle dataset called h2-buildings (used by T2 and T3).")